### Ingest Consignors CSV
This notebook ingests the consignors file from the landing layer and writes it to a bronze Delta table.

Business context:
* This table identifies the individuals or entities who consign items for Company auctions
* Each row represents one consignor record with their email, deal/consignment code, and province
* The bronze table preserves these records for downstream analysis of consignor coverage, auction sourcing, and regional participation

#### Step 1: Load shared setup values
This step imports the common project configuration used across bronze ingestion notebooks.

What happens here:
* Loads shared Unity Catalog catalog, schema, and volume variables
* Ensures consistent environment settings across all bronze ingestion notebooks
* Keeps environment-specific values centralized so changes propagate everywhere


#### Step 2: Build the consignors file path
This step assembles the full storage path for the raw consignors CSV.

What happens here:
* Accepts a `batch_id` widget parameter (date string, e.g. `2024-02-06`) to locate the correct batch folder
* Combines the volume, catalog, schema, folder, and batch identifier into a single path
* Prints the resolved path so it can be validated before reading data


#### Step 3: Import helper libraries
This step imports the PySpark functions module used in subsequent transformation and enrichment steps.

What happens here:
* Makes `pyspark.sql.functions` available as `F` for use in DataFrame operations
* Required before applying any column expressions or metadata additions downstream


#### Consignors data dictionary
This file represents individuals or entities who consign lots for Company auctions, where each row identifies one consignor record.

Key fields:
* `email`: the consignor's email address, used as the primary identifier
* `dcc`: the deal or consignment code that groups the consignor under a specific auction or sourcing arrangement (e.g. `WFP`, `MM`, `JP`, `EL`, `Legacy`)
* `consignor_province`: the Canadian province associated with the consignor

Business use:
* Identify who is sourcing inventory for each auction batch
* Analyze consignor participation by region and deal code
* Support downstream auction lot attribution and consignor reporting workflows


#### Step 4: Load the consignors data
This step reads the raw consignors CSV into a Spark DataFrame.

What happens here:
* Reads the CSV file at the resolved batch path with a header row
* Uses schema inference to detect column types automatically
* Produces the `df_consignors` DataFrame used for all subsequent steps


#### Step 5: Preview the consignors data
This step displays the loaded DataFrame for a quick visual check before writing to the bronze layer.

What happens here:
* Renders the full consignors dataset as an interactive table
* Allows row counts, column values, and data quality to be verified at a glance
* Acts as a lightweight validation gate before the bronze write


#### Step 6: Write the bronze consignors table
This final step persists the consignors DataFrame as a managed Delta table in the bronze schema.

What happens here:
* Writes the ingested consignors data to `{catalog}.{bronze}.consignors`
* Uses overwrite mode so the table is fully rebuilt on each run
* Makes the bronze dataset available for downstream silver transformations, consignor reporting, and auction sourcing analysis


In [ ]:
catalog = dbutils.widgets.get("catalog")
container = dbutils.widgets.get("container")
storage_account = dbutils.widgets.get("storage_account")

In [ ]:
%run "../setup/002_Environment_variables"

In [ ]:
dbutils.widgets.text('id'," ")
batch_id = dbutils.widgets.get('id')
batch_id

In [ ]:
# Build the full source path for the CSV file to ingest from cloud storage.
# This keeps the input file location parameterized through shared environment values.
path = f'/{volume}/{catalog}/{schema}/{file}/{batch_id}/consignors.csv'

In [ ]:
df_consignors = spark.read.format('csv') \
    .option('header', 'true') \
         .option('multiLine', 'true') \
    .option("quote", '"') \
    .option("escape", '"') \
        .option('timestampFormat', 'yyyy-MM-dd HH:mm:ss z') \
    .option('inferSchema', 'True') \
    .load(path)

In [ ]:
df_consignors = df_consignors.toDF(*[column_name.strip() for column_name in df_consignors.columns])

In [ ]:
df_consignors = df_consignors.select("email",'dcc','province')

In [ ]:
df_consignors = df_consignors.dropDuplicates(["email"])

In [ ]:
# Create or replace the silver Delta table that stores curated lot-level auction results.
# This schema supports reporting on sold vs unsold lots, solid vs mixed lots, and realized pricing.

spark.sql(
f"""
CREATE TABLE IF NOT EXISTS {catalog}.bronze.consignors (
    email STRING,
    dcc STRING,
    province STRING
)
USING DELTA
LOCATION 'abfss://{container}@{storage_account}.dfs.core.windows.net/bronze/consignors' """)

In [ ]:
df_consignors.createOrReplaceTempView('df_consignors_view')

In [ ]:

spark.sql(
f""" 
MERGE INTO {catalog}.bronze.consignors AS target
USING df_consignors_view AS source
ON target.email = source.email
WHEN MATCHED THEN
UPDATE SET
  target.dcc = source.dcc,
  target.email = source.email,
  target.province = source.province
WHEN NOT MATCHED THEN
INSERT (
  email, dcc, province
)
VALUES (
  source.email, source.dcc, source.province
)
"""
)